<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [ ]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [1]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

Warm-up comparison (completed with assistant assistance):

Weak prompt: "add a search feature". Assistant response in this follow-up: "What should be searched, and how should results look?" This is a new illustrative response, not a recovered earlier exchange.

Project-aware prompt: "Read AGENTS.md, then propose a plan to add an optional tags filter to search_documents in src/bootcamp_agent/tools.py. Only that file may change. No new dependencies. Plan only, no edits."

Actual earlier response excerpt: "Add tags: Sequence[str] | None = None after max_results, preserving existing calls." It also specified exact case-sensitive any-tag matching and filtering after retrieval.

Difference: the scoped prompt supplies the file, function, dependency restriction and plan-before-edit boundary; the vague prompt leaves those decisions open.

## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason (learner's exact words):** I reject adding a new dependency because Python's built-in features handle the tags filter, and the approved scope explicitly says no new dependencies.

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

Added a Session 1 review boundary to AGENTS.md: "For scoped feature work, keep the result cap and dependency list unchanged; record assistant checks separately from learner review, and never present an unconfirmed learner action as completed." This addresses the difference between assistant verification and learner evidence.

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [2]:
loop = {'plan_approved': 'The learner approved the plan in the preceding chat: edit only tools.py, add optional tags after max_results, validate inputs, match any exact case-sensitive tag after retrieval, and retain the existing cap and citations.', 'diff_inspected': 'Assistant review: inspected the tools.py diff, including the signature, tag validation, post-retrieval filtering, and tool description. Ruff and offline behavioral checks passed. The learner received the diff and explanation; independent line-by-line learner review has not been explicitly confirmed.', 'rejected_change': 'The learner rejected the proposed addition of a new dependency for the tags filter; no dependency was installed or added.', 'why_rejected': "I reject adding a new dependency because Python's built-in features handle the tags filter, and the approved scope explicitly says no new dependencies.", 'risks': 'Filtering happens after the capped retrieval, so matching documents below the cutoff are omitted and results may be fewer or empty. Tags match exactly and case-sensitively; whitespace is not normalized. The ch01 evidence check validates the log shape, not the feature or independent learner review.'}
for key, value in loop.items():
    print(f"{key:18} {value[:58]}")


plan_approved      The learner approved the plan in the preceding chat: edit 
diff_inspected     Assistant review: inspected the tools.py diff, including t
rejected_change    The learner rejected the proposed addition of a new depend
why_rejected       I reject adding a new dependency because Python's built-in
risks              Filtering happens after the capped retrieval, so matching 


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [3]:
check("ch01-e1", loop)

✅ ch01-e1 passed


## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [4]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.

## Verification and provenance

The assistant inspected the existing implementation and ran Ruff successfully. Offline checks passed for unchanged calls, empty tags, matching and nonmatching tags, multiple tags, case sensitivity, invalid arguments, cap preservation, order/citations, and the post-retrieval cutoff. The learner supplied the rejection quoted above. The notebook does not claim independent learner review that was not confirmed.

Only the import, loop, check and review cells were executed during this submission preparation; no live model or API was needed.

Reference: Python built-in `any`: https://docs.python.org/3.10/library/functions.html#any